# 11 - Interpretability with Integrated Gradients


## Notes

Train an end-to-end `ArcFaceModel` with an unfrozen EfficientNetB3 backbone and visualize full-model Integrated Gradients heatmaps on validation images.

- The notebook trains the complete model directly on images and updates the backbone weights jointly with the ArcFace projection head.
- If a matching checkpoint already exists, the training cell skips training and loads that checkpoint instead.
- The attribution target is the cosine class score induced by the trained ArcFace weights, not the margin-modified training logit.
- The interpretability section uses `captum.attr.IntegratedGradients`. If Captum is not installed in your environment, run `%pip install captum` in a notebook cell first.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm.data
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from dotenv import load_dotenv
from torch.utils.data import DataLoader

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.training as training
import src.wandb_utils as wandb_utils
from src.interpretability import (
    ArcFaceClassifierForAttribution,
    attribution_to_heatmap,
)
from src.models import ArcFaceModel
from src.utils import get_device, set_seed

load_dotenv(project_root / '.env')

RANDOM_SEED = 42
set_seed(RANDOM_SEED)
device = get_device()
device


## Config

In [ ]:
config = {
    'data_dir': Path('../data'),
    'checkpoint_dir': Path('../checkpoints/e6_interpretability'),
    'results_dir': Path('../interpretability_results/11_interpretability'),
    'backbone_name': 'efficientnet_b3',
    'input_size': 300,
    'embedding_dim': 256,
    'hidden_dim': 512,
    'dropout': 0.3,
    'freeze_backbone': False,
    'arcface_margin': 0.5,
    'arcface_scale': 64.0,
    'batch_size': 16,
    'learning_rate': 1e-4,
    'head_learning_rate': 1e-4,
    'backbone_learning_rate': 1e-5,
    'weight_decay': 1e-4,
    'num_epochs': 30,
    'patience': 5,
    'val_split': 0.2,
    'num_workers': 2,
    'seed': RANDOM_SEED,
    'checkpoint_name': 'efficientnet_b3_arcface_end_to_end.pth',
    'load_checkpoint_if_available': True,
}

config['checkpoint_dir'].mkdir(parents=True, exist_ok=True)
config['results_dir'].mkdir(parents=True, exist_ok=True)
config


## Load Data Split

In [ ]:
train_df = data.load_train_df(config['data_dir'])
train_df, label_encoder = data.encode_labels(train_df)
num_classes = len(label_encoder.classes_)

train_data, val_data = data.train_val_split(
    train_df,
    val_split=config['val_split'],
    seed=config['seed'],
    stratify_col='ground_truth',
)

print(f'Train samples: {len(train_data)}')
print(f'Val samples:   {len(val_data)}')
print(f'Num classes:   {num_classes}')

## Build End-To-End Model

In [ ]:
model = ArcFaceModel(
    backbone_name=config['backbone_name'],
    num_classes=num_classes,
    embedding_dim=config['embedding_dim'],
    hidden_dim=config['hidden_dim'],
    margin=config['arcface_margin'],
    scale=config['arcface_scale'],
    dropout=config['dropout'],
    pretrained=True,
    freeze_backbone=config['freeze_backbone'],
).to(device)

data_config = timm.data.resolve_model_data_config(model.backbone)
mean = data_config['mean']
std = data_config['std']

train_loader, val_loader = data.create_dataloaders(
    train_df=train_data,
    val_df=val_data,
    img_dir=config['data_dir'] / 'train' / 'train',
    input_size=config['input_size'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    mean=mean,
    std=std,
    weighted_sampling=False,
    label_col='label_encoded',
    augment=False,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')


## Train ArcFaceModel

## W&B Logging


In [ ]:
backbone_param_count = sum(p.numel() for p in model.backbone.parameters())
param_count = sum(p.numel() for p in model.parameters())
run_name = '11-EfficientNetB3-ArcFace-EndToEnd-IntegratedGradients'
wandb = wandb_utils.init_wandb(
    config,
    run_name=run_name,
    param_count=param_count,
    param_count_backbone=backbone_param_count,
)
wandb


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = training.build_optimizer(model, config)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

checkpoint_name = config['checkpoint_name']
checkpoint_path = config['checkpoint_dir'] / checkpoint_name

if config.get('load_checkpoint_if_available', False) and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    results = {
        'best_map': checkpoint.get('val_map'),
        'best_val_loss': checkpoint.get('val_loss'),
        'best_epoch': checkpoint.get('epoch'),
        'epochs_ran': checkpoint.get('epoch'),
    }
    print(f'Loaded checkpoint from {checkpoint_path}')
else:
    results = training.fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

wandb.run.summary['best_val_mAP'] = results['best_map']
wandb.run.summary['best_val_loss'] = results['best_val_loss']
wandb.run.summary['best_epoch'] = results['best_epoch']
wandb.run.summary['total_epochs'] = results['epochs_ran']

wandb.finish()

results['best_map'], results['best_epoch'], results['epochs_ran']


## Training Curves

In [ ]:
if not config.get('load_checkpoint_if_available', False):
    history = pd.DataFrame(results['history'])
    history.index = np.arange(1, len(history) + 1)
    history.index.name = 'epoch'
    history.tail()

In [ ]:
if not config.get('load_checkpoint_if_available', False):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(history.index, history['train_loss'], label='train')
    axes[0].plot(history.index, history['val_loss'], label='val')
    axes[0].set_title('Loss')
    axes[0].legend()

    axes[1].plot(history.index, history['train_acc'], label='train')
    axes[1].plot(history.index, history['val_acc'], label='val')
    axes[1].set_title('Accuracy')
    axes[1].legend()

    axes[2].plot(history.index, history['val_map'], label='val mAP', color='tab:green')
    axes[2].set_title('Validation mAP')
    axes[2].legend()

    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

## Build Full-Model Classifier For Attribution


In [ ]:
classifier = ArcFaceClassifierForAttribution(model).to(device)
classifier.eval()


## Captum Setup


In [ ]:
try:
    from captum.attr import IntegratedGradients
except ImportError as exc:
    raise ImportError(
        'Captum is required for Integrated Gradients. Install it with `%pip install captum` and rerun this notebook.'
    ) from exc


ig = IntegratedGradients(classifier)


## Attribution Helpers


In [ ]:
image_dir = config['data_dir'] / 'train' / 'train'
transform = data.build_transforms_baseline(
    input_size=config['input_size'],
    train=False,
    mean=mean,
    std=std,
    augment=False,
)


def load_image_tensor(filename):
    image = Image.open(image_dir / filename).convert('RGB')
    tensor = transform(image)
    return image, tensor


def predict_row(row):
    _, tensor = load_image_tensor(row['filename'])
    logits = classifier(tensor.unsqueeze(0).to(device))
    pred_idx = int(logits.argmax(dim=1).item())
    pred_score = float(logits.max(dim=1).values.item())
    return pred_idx, pred_score


def compute_attribution_result(filename, target_idx=None, attribution_model=None):
    image, tensor = load_image_tensor(filename)
    input_batch = tensor.unsqueeze(0).to(device)
    input_batch.requires_grad_(True)

    current_classifier = classifier if attribution_model is None else attribution_model
    logits = current_classifier(input_batch)
    if target_idx is None:
        target_idx = int(logits.argmax(dim=1).item())

    current_ig = IntegratedGradients(current_classifier)
    attribution = current_ig.attribute(
        input_batch,
        target=target_idx,
        baselines=torch.zeros_like(input_batch),
    )
    heatmap = attribution_to_heatmap(attribution)[0].detach().cpu().numpy()
    return {
        'filename': filename,
        'image': image,
        'heatmap': heatmap,
        'pred_idx': target_idx,
        'pred_label': label_encoder.inverse_transform([target_idx])[0],
        'pred_score': float(logits[0, target_idx].item()),
    }


def show_attribution_result(result, ax_image, ax_heatmap):
    image_np = np.asarray(result['image'])
    heatmap = result['heatmap']
    heatmap_image = Image.fromarray(np.uint8(heatmap * 255)).resize(
        result['image'].size,
        resample=Image.BILINEAR,
    )
    heatmap_resized = np.asarray(heatmap_image, dtype=np.float32) / 255.0

    ax_image.imshow(image_np)
    ax_image.set_title(
        f"Pred: {result['pred_label']}\nscore={result['pred_score']:.3f}"
    )
    ax_image.axis('off')

    ax_heatmap.imshow(image_np)
    ax_heatmap.imshow(heatmap_resized, cmap='inferno', alpha=0.45)
    ax_heatmap.set_title('Integrated Gradients')
    ax_heatmap.axis('off')


def save_attribution_figure(fig, filename):
    output_path = config['results_dir'] / filename
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved {output_path}')
    return output_path


## Score Validation Images

In [ ]:
val_examples = val_data.copy().reset_index(drop=True)
preds = val_examples.apply(predict_row, axis=1, result_type='expand')
preds.columns = ['pred_idx', 'pred_score']
preds['pred_idx'] = preds['pred_idx'].astype(int)
preds['pred_score'] = preds['pred_score'].astype(float)

val_examples = pd.concat([val_examples, preds], axis=1)
val_examples['pred_label'] = label_encoder.inverse_transform(val_examples['pred_idx'].to_numpy())
val_examples['is_correct'] = val_examples['pred_idx'] == val_examples['label_encoded']

print(f'Validation top-1 accuracy on scored samples: {val_examples["is_correct"].mean():.3f}')
val_examples[['filename', 'ground_truth', 'pred_label', 'pred_score', 'is_correct']].head()


## Integrated Gradients On Correct Validation Predictions


In [ ]:
selected_examples = (
    val_examples[val_examples['is_correct']]
    .sort_values(['pred_score', 'filename'], ascending=[False, True])
    .head(6)
    .reset_index(drop=True)
)

selected_examples[['filename', 'ground_truth', 'pred_label', 'pred_score']]

In [ ]:
fig, axes = plt.subplots(len(selected_examples), 2, figsize=(10, 4 * len(selected_examples)))
if len(selected_examples) == 1:
    axes = np.array([axes])

for row_idx, row in selected_examples.iterrows():
    result = compute_attribution_result(row['filename'], target_idx=int(row['pred_idx']))
    show_attribution_result(result, axes[row_idx, 0], axes[row_idx, 1])

plt.tight_layout()
save_attribution_figure(fig, 'correct_predictions_integrated_gradients.png')
plt.show()


## Optional: Integrated Gradients On Misclassifications


In [ ]:
mistakes = (
    val_examples[~val_examples['is_correct']]
    .sort_values(['pred_score', 'filename'], ascending=[False, True])
    .head(4)
    .reset_index(drop=True)
)
mistakes[['filename', 'ground_truth', 'pred_label', 'pred_score']]

In [ ]:
if len(mistakes) > 0:
    fig, axes = plt.subplots(len(mistakes), 2, figsize=(10, 4 * len(mistakes)))
    if len(mistakes) == 1:
        axes = np.array([axes])

    for row_idx, row in mistakes.iterrows():
        result = compute_attribution_result(row['filename'], target_idx=int(row['pred_idx']))
        show_attribution_result(result, axes[row_idx, 0], axes[row_idx, 1])

    plt.tight_layout()
    save_attribution_figure(fig, 'misclassifications_integrated_gradients.png')
    plt.show()
else:
    print('No validation mistakes found for the selected split.')


## Sanity Check: Randomized Weights


Randomize the trained model weights and rerun Integrated Gradients on the same examples. If the method is meaningful, the heatmaps should become much less structured and informative.


In [ ]:
import copy


def randomize_module_weights(module):
    if hasattr(module, 'reset_parameters'):
        module.reset_parameters()


randomized_model = copy.deepcopy(model).to(device)
randomized_model.apply(randomize_module_weights)
randomized_model.eval()

randomized_classifier = ArcFaceClassifierForAttribution(randomized_model).to(device)
randomized_classifier.eval()


In [ ]:
def compute_randomized_attribution_result(filename, target_idx=None):
    return compute_attribution_result(
        filename,
        target_idx=target_idx,
        attribution_model=randomized_classifier,
    )


In [ ]:
fig, axes = plt.subplots(len(selected_examples), 3, figsize=(15, 4 * len(selected_examples)))
if len(selected_examples) == 1:
    axes = np.array([axes])

for row_idx, row in selected_examples.iterrows():
    trained_result = compute_attribution_result(row['filename'], target_idx=int(row['pred_idx']))
    randomized_result = compute_randomized_attribution_result(row['filename'], target_idx=int(row['pred_idx']))

    axes[row_idx, 0].imshow(np.asarray(trained_result['image']))
    axes[row_idx, 0].set_title(trained_result['filename'])
    axes[row_idx, 0].axis('off')

    axes[row_idx, 1].imshow(np.asarray(trained_result['image']))
    axes[row_idx, 1].imshow(trained_result['heatmap'], cmap='inferno', alpha=0.45)
    axes[row_idx, 1].set_title('IG trained model')
    axes[row_idx, 1].axis('off')

    axes[row_idx, 2].imshow(np.asarray(randomized_result['image']))
    axes[row_idx, 2].imshow(randomized_result['heatmap'], cmap='inferno', alpha=0.45)
    axes[row_idx, 2].set_title('IG randomized model')
    axes[row_idx, 2].axis('off')

plt.tight_layout()
save_attribution_figure(fig, 'randomized_weights_sanity_check_integrated_gradients.png')
plt.show()


## Masking Faithfulness Test (Similarity)


Use the Integrated Gradients heatmap on the query image to mask the top-k most relevant pixels, recompute embedding similarity, and compare the drop against masking the same number of random pixels.


In [ ]:
masking_config = {
    'top_fraction': 0.10,
    'num_true_pairs': 5,
    'num_impostor_pairs': 5,
    'random_seed': RANDOM_SEED,
}
masking_config


In [ ]:
def compute_embedding_from_tensor(tensor):
    with torch.no_grad():
        embedding = model.get_embeddings(tensor.unsqueeze(0).to(device))[0].detach().cpu()
    return embedding


def cosine_similarity_from_tensors(query_tensor, gallery_tensor):
    query_embedding = compute_embedding_from_tensor(query_tensor)
    gallery_embedding = compute_embedding_from_tensor(gallery_tensor)
    return float(torch.dot(query_embedding, gallery_embedding).item())


def build_top_mask(heatmap, top_fraction):
    flat = heatmap.reshape(-1)
    k = max(1, int(len(flat) * top_fraction))
    threshold = np.partition(flat, -k)[-k]
    return heatmap >= threshold


def apply_mask_to_tensor(tensor, mask, fill_value=0.0):
    masked = tensor.clone()
    mask_tensor = torch.from_numpy(mask).to(masked.device)
    masked[:, mask_tensor] = fill_value
    return masked


def apply_random_mask_to_tensor(tensor, num_pixels, rng, fill_value=0.0):
    masked = tensor.clone()
    flat_mask = np.zeros(masked.shape[-2] * masked.shape[-1], dtype=bool)
    random_indices = rng.choice(len(flat_mask), size=num_pixels, replace=False)
    flat_mask[random_indices] = True
    random_mask = flat_mask.reshape(masked.shape[-2], masked.shape[-1])
    masked[:, torch.from_numpy(random_mask).to(masked.device)] = fill_value
    return masked, random_mask


def build_similarity_pairs(val_examples, num_true_pairs=5, num_impostor_pairs=5):
    correct_examples = val_examples[val_examples['is_correct']].copy()

    true_pairs = []
    for label, group in correct_examples.groupby('ground_truth'):
        if len(group) < 2:
            continue
        group = group.sort_values(['pred_score', 'filename'], ascending=[False, True]).head(2)
        query_row, gallery_row = group.iloc[0], group.iloc[1]
        true_pairs.append({
            'pair_type': 'true_match',
            'query_filename': query_row['filename'],
            'gallery_filename': gallery_row['filename'],
            'query_label': query_row['ground_truth'],
            'gallery_label': gallery_row['ground_truth'],
            'target_idx': int(query_row['pred_idx']),
        })
        if len(true_pairs) >= num_true_pairs:
            break

    impostor_pairs = []
    sorted_examples = correct_examples.sort_values(['pred_score', 'filename'], ascending=[False, True]).reset_index(drop=True)
    for idx, query_row in sorted_examples.iterrows():
        for jdx, gallery_row in sorted_examples.iloc[idx + 1 :].iterrows():
            if query_row['ground_truth'] == gallery_row['ground_truth']:
                continue
            impostor_pairs.append({
                'pair_type': 'impostor',
                'query_filename': query_row['filename'],
                'gallery_filename': gallery_row['filename'],
                'query_label': query_row['ground_truth'],
                'gallery_label': gallery_row['ground_truth'],
                'target_idx': int(query_row['pred_idx']),
            })
            break
        if len(impostor_pairs) >= num_impostor_pairs:
            break

    return pd.DataFrame(true_pairs + impostor_pairs)


In [ ]:
pair_df = build_similarity_pairs(
    val_examples,
    num_true_pairs=masking_config['num_true_pairs'],
    num_impostor_pairs=masking_config['num_impostor_pairs'],
)
pair_df


In [ ]:
rng = np.random.default_rng(masking_config['random_seed'])
masking_results = []

for _, pair in pair_df.iterrows():
    _, query_tensor = load_image_tensor(pair['query_filename'])
    _, gallery_tensor = load_image_tensor(pair['gallery_filename'])
    attribution_result = compute_attribution_result(pair['query_filename'], target_idx=int(pair['target_idx']))

    important_mask = build_top_mask(attribution_result['heatmap'], masking_config['top_fraction'])
    num_masked_pixels = int(important_mask.sum())

    masked_query_tensor = apply_mask_to_tensor(query_tensor, important_mask, fill_value=0.0)
    random_query_tensor, random_mask = apply_random_mask_to_tensor(
        query_tensor,
        num_pixels=num_masked_pixels,
        rng=rng,
        fill_value=0.0,
    )

    original_similarity = cosine_similarity_from_tensors(query_tensor, gallery_tensor)
    important_similarity = cosine_similarity_from_tensors(masked_query_tensor, gallery_tensor)
    random_similarity = cosine_similarity_from_tensors(random_query_tensor, gallery_tensor)

    masking_results.append({
        'pair_type': pair['pair_type'],
        'query_filename': pair['query_filename'],
        'gallery_filename': pair['gallery_filename'],
        'query_label': pair['query_label'],
        'gallery_label': pair['gallery_label'],
        'target_idx': int(pair['target_idx']),
        'masked_fraction': num_masked_pixels / important_mask.size,
        'original_similarity': original_similarity,
        'important_mask_similarity': important_similarity,
        'random_mask_similarity': random_similarity,
        'important_drop': original_similarity - important_similarity,
        'random_drop': original_similarity - random_similarity,
    })

masking_results_df = pd.DataFrame(masking_results)
masking_results_df


In [ ]:
summary_df = (
    masking_results_df.groupby('pair_type')[['important_drop', 'random_drop']]
    .mean()
    .reset_index()
)
summary_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pair_plot_df = masking_results_df.copy()
pair_plot_df['pair_name'] = pair_plot_df['query_filename'] + ' -> ' + pair_plot_df['gallery_filename']

x = np.arange(len(pair_plot_df))
width = 0.35
axes[0].bar(x - width / 2, pair_plot_df['important_drop'], width=width, label='important mask')
axes[0].bar(x + width / 2, pair_plot_df['random_drop'], width=width, label='random mask')
axes[0].set_xticks(x)
axes[0].set_xticklabels(pair_plot_df['pair_name'], rotation=75, ha='right')
axes[0].set_ylabel('Similarity drop')
axes[0].set_title('Per-pair similarity drop after masking')
axes[0].legend()

summary_plot = summary_df.set_index('pair_type')
summary_plot[['important_drop', 'random_drop']].plot(kind='bar', ax=axes[1])
axes[1].set_ylabel('Mean similarity drop')
axes[1].set_title('Mean drop by pair type')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
save_attribution_figure(fig, 'masking_faithfulness_similarity.png')
plt.show()


In [ ]:
masking_results_path = config['results_dir'] / 'masking_faithfulness_similarity.csv'
masking_results_df.to_csv(masking_results_path, index=False)
print(masking_results_path)


## Summary

This notebook now matches the requested end-to-end setup:

1. `ArcFaceModel` is trained directly on images with `freeze_backbone=False`.
2. EfficientNetB3 backbone weights are updated jointly with the projection head.
3. Integrated Gradients is applied to a full-model image-to-class-score wrapper.
4. Attribution therefore flows through the entire backbone instead of stopping at cached embeddings.
5. A masking faithfulness test measures whether masking top-importance pixels reduces embedding similarity more than masking a random region of the same size.
